In [1]:
import os
import pandas as pd
import rasterio
import numpy as np

1. Chargement des coordonées GPS de tous les pays
2. Lecture du fichier .tif, transformation des coordonées GPS en coordonnées pixels 
3. Trouver les valeurs des radiances associées à ces coordonnées pixels
4. Insertion de la colonne radiance dans une dataframe en l'associant à ces valeurs correspondantes en latitude, longitude et country
5. Sauvegarde

In [2]:


# Chargement du fichier CSV
df = pd.read_csv("D:\df_afrique_pharmacies_schools_combined_sampled.csv") 


# Chargement du fichier TIFF (Nighttime light 2023)
tif_file_path = r"G:\VNL_npp_2023_global_vcmslcfg_v2_c202402081600.average.dat.tif\VNL_npp_2023_global_vcmslcfg_v2_c202402081600.average.dat.tif"
dataset = rasterio.open(tif_file_path)

# Lire les données du raster en mémoire une seule fois
raster_data = dataset.read(1)

# Transformation inverse
transform = dataset.transform

# Transformation des coordonnées GPS en indices de pixels (vectorisé)
def gps_to_pixel(lon, lat, transform):
    # Transformation des coordonnées GPS en indices de pixels
    col, row = ~transform * (lon, lat)
    return int(col), int(row)

# Fonction pour récupérer la valeur de luminance
def get_radiance_value_vectorized(lon, lat, raster_data, transform):
    col, row = gps_to_pixel(lon, lat, transform)
    
    # Vérification si l'indice est dans les limites de l'image
    if (0 <= row < raster_data.shape[0]) and (0 <= col < raster_data.shape[1]):
        return raster_data[row, col]
    else:
        return None  # Si hors de l'image

# Appliquer la fonction de manière vectorisée
df['radiance'] = df.apply(lambda row: get_radiance_value_vectorized(row['longitude'], row['latitude'], raster_data, transform), axis=1)




In [ ]:
# Affichage du nouveau dataframe
print(df)

In [ ]:
missing_values = df['radiance'].isnull().sum()
print(f"Nombre de valeurs manquantes dans 'radiance' : {missing_values}")

In [7]:
# Enregistrement du résultat dans un nouveau fichier CSV 
output_csv_path = r"D:\df_afrique_pharmacies_schools_combined_sampled.csv"
df.to_csv(output_csv_path, index=False)

  1.Importation des bibliothèques nécessaires
  2.Calcul des statistiques descriptives et du résumé en cinq nombres
  3.Visualisation des données avec des histogrammes et des boxplots
  Analyse supplémentaire (optionnelle)
  4.Enregistrement des résultats et des graphiques

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [10]:
# Optionnel : pour une meilleure apparence des graphiques
sns.set(style="whitegrid")

In [11]:
sns.set_style('whitegrid')

In [ ]:
# Statistiques descriptives
descriptive_stats = df['radiance'].describe()
print("Statistiques descriptives :")
print(descriptive_stats)

# Résumé en cinq nombres
five_number_summary = df['radiance'].describe()[['min', '25%', '50%', '75%', 'max']]
print("\nRésumé en cinq nombres :")
print(five_number_summary)

In [ ]:
# Calcul de mesures supplémentaires
print("\nMesures supplémentaires:")
variance = df['radiance'].var()
mode = df['radiance'].mode()[0]
skewness = df['radiance'].skew()
kurtosis = df['radiance'].kurtosis()

print(f"Variance: {variance:.2f}")
print(f"Mode: {mode}")
print(f"Asymétrie (Skewness): {skewness:.2f}")
print(f"Applatissement (Kurtosis): {kurtosis:.2f}")



In [ ]:
percentile_99 = df['radiance'].quantile(0.999)
print(f"Le 99 ème percentile de la colonne 'radiance' est : {percentile_99}")  #La valeur qui dépasse 99.9% de toutes les valeurs

#### On supprime toutes les valeurs de df qui sont au-dessus du 99.9ème percentile de la colonne radiance

In [ ]:
import pandas as pd

# Calcul du 99ème percentile
percentile_99 = df['radiance'].quantile(0.999)
print(f"Le 99ème percentile de la colonne 'radiance' est : {percentile_99}")

# Filtrer le DataFrame pour ne garder que les valeurs de 'radiance' inférieures ou égales au 99ème percentile
initial_count = len(df)
df = df[df['radiance'] <= percentile_99]
filtered_count = len(df)

# Calculer et imprimer le nombre de valeurs supprimées
num_values_removed = initial_count - filtered_count
print(f"Le nombre de valeurs supprimées est : {num_values_removed}")

# Afficher le DataFrame mis à jour
print(df)


In [ ]:
# Statistiques descriptives
descriptive_stats = df['radiance'].describe()
print("Statistiques descriptives :")
print(descriptive_stats)

# Résumé en cinq nombres
five_number_summary = df['radiance'].describe()[['min', '25%', '50%', '75%', 'max']]
print("\nRésumé en cinq nombres :")
print(five_number_summary)

In [ ]:
# Calcul de mesures supplémentaires
print("\nMesures supplémentaires:")
variance = df['radiance'].var()
mode = df['radiance'].mode()[0]
skewness = df['radiance'].skew()
kurtosis = df['radiance'].kurtosis()

print(f"Variance: {variance:.2f}")
print(f"Mode: {mode}")
print(f"Asymétrie (Skewness): {skewness:.2f}")
print(f"Applatissement (Kurtosis): {kurtosis:.2f}")



In [ ]:
df.shape

In [72]:
# Enregistrement du résultat dans un nouveau fichier CSV 
output_csv_path = r"D:\df_afrique_pharmacies_schools_combined_sampled_cleaned.csv"
df.to_csv(output_csv_path, index=False)

##### Visualisation des données

Histogramme:Un histogramme permet de visualiser la distribution des valeurs de radiance.
Boxplot:Un boxplot permet d'identifier les valeurs aberrantes et de visualiser la distribution des données.

In [ ]:
#Histogramme
plt.figure(figsize=(6, 10))
sns.histplot(df['radiance'], bins=60, kde=True, color='blue')
plt.title('Distribution des valeurs de Radiance')
plt.xlabel('Radiance')
plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()


In [ ]:
#Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(x=df['radiance'], color='green')
plt.title('Boxplot des valeurs de Radiance')
plt.xlabel('Radiance')
plt.tight_layout()
plt.show()


In [ ]:
#Carte de Chaleur (Heatmap) de la Radiance
# Création d'une carte de chaleur si les données sont géospatiales
#La carte de chaleur représente la distribution spatiale des intensités lumineuses.
#Les couleurs plus chaudes indiquent des zones avec une radiance plus élevée, tandis que les couleurs plus froides indiquent une radiance plus faible.
#Cela peut être utile pour identifier des zones urbaines ou des points chauds spécifiques.

plt.figure(figsize=(10,10))
scatter = plt.scatter(df['longitude'], df['latitude'], c=df['radiance'], cmap='inferno', alpha=0.7)
plt.colorbar(scatter, label='Radiance')
plt.title('Carte de Chaleur des Intensités Lumineuses Nocturnes')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [40]:
#### conda install geopandas
#### conda install -c conda-forge contextily


In [ ]:
import folium
from folium.plugins import HeatMap

# Créer une carte de base centrée sur une région
m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=6)

# Ajouter les points de chaleur
heat_data = [[row['latitude'], row['longitude'], row['radiance']] for index, row in df.iterrows()]
HeatMap(heat_data).add_to(m)

# Afficher la carte
m.save("heatmap_radiance.html")
m

###### Sauvegarde de la carte

In [ ]:
!pip install selenium pillow


In [ ]:
!pip install webdriver-manager

In [ ]:
import folium
from folium.plugins import HeatMap
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from PIL import Image
import time

# Créer une carte de base centrée sur une région
m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=4)

# Ajouter les points de chaleur
heat_data = [[row['latitude'], row['longitude'], row['radiance']] for index, row in df.iterrows()]
HeatMap(heat_data).add_to(m)

# Sauvegarder la carte au format HTML
html_file_path = r"D:\heatmap_radiance.html"
m.save(html_file_path)

# Utiliser Selenium pour ouvrir la carte et prendre une capture d'écran
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Créer le service pour le driver Chrome
service = Service(ChromeDriverManager().install())

# Initialiser le driver avec le service
driver = webdriver.Chrome(service=service, options=options)
driver.set_window_size(1000, 1000)

# Charger la carte HTML
driver.get(f"file:///{html_file_path}")

# Attendre que la carte se charge complètement
time.sleep(3)

# Sauvegarder une capture d'écran
png_file_path = r"D:\heatmap_African_school_GPS_locations.png"
driver.save_screenshot(png_file_path)

# Fermer le navigateur
driver.quit()

# Convertir en JPEG si nécessaire
image = Image.open(png_file_path)
jpeg_file_path = r"D:\heatmap_African_school_GPS_locations.jpeg"
image = image.convert("RGB")
image.save(jpeg_file_path, "JPEG")


In [ ]:
#Densités
#Une courbe de densité peut offrir une autre perspective sur la distribution des données.
plt.figure(figsize=(10, 6))
sns.kdeplot(df['radiance'], shade=True, color='purple')
plt.title('Courbe de Densité des valeurs de Radiance')
plt.xlabel('Radiance')
plt.ylabel('Densité')
plt.tight_layout()
plt.show()


In [ ]:
### Analyse de la Distribution
#Le test de Shapiro-Wilk est utilisé pour vérifier si les données suivent une distribution normale.
#Une p-value supérieure à 0.05 indique que les données sont normalement distribuées.

# Vérification de la normalité de la distribution
from scipy.stats import shapiro

stat, p = shapiro(df['radiance'].dropna())
print('Statistique de test Shapiro-Wilk: {:.3f}, p-value: {:.3f}'.format(stat, p))

if p > 0.05:
    print("Les données suivent une distribution normale (à un niveau de signification de 5%).")
else:
    print("Les données ne suivent pas une distribution normale (à un niveau de signification de 5%).")


Étape 1 : Détection des Outliers et Affichage de leurs Coordonnées et Pays

In [ ]:
# Calculer l'IQR pour détecter les outliers
Q1 = df['radiance'].quantile(0.25)
Q3 = df['radiance'].quantile(0.75)
IQR = Q3 - Q1

# Définir les limites pour les outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrer les outliers
outliers = df[(df['radiance'] < lower_bound) | (df['radiance'] > upper_bound)]

# Afficher la latitude, longitude et le pays des outliers
print("Liste des outliers avec leur latitude, longitude, et radiance :")
print(outliers[['latitude', 'longitude','radiance']])


In [ ]:
lower_bound

In [ ]:
upper_bound

Mélange Gaussien

On va considérer le cas avec outliers (3 classes et n classes optimales) ainsi que le cas sans outlier (3 classes et n classes optimales)

#### Avec les outliers

In [80]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns

def display_class_ranges_sorted(gmm, data, column):
    """Affiche les plages de valeurs des classes, triées par valeur minimale."""
    data['class'] = gmm.predict(data[column].values.reshape(-1, 1))
    class_ranges = []

    for i in range(gmm.n_components):
        class_values = data[data['class'] == i][column]
        class_ranges.append((i, class_values.min(), class_values.max(), len(class_values)))

    # Trier les classes par valeur minimale
    class_ranges_sorted = sorted(class_ranges, key=lambda x: x[1])

    # Afficher les informations des classes triées
    for idx, (class_index, min_val, max_val, count) in enumerate(class_ranges_sorted, start=1):
        print(f"Classe {idx}: Min = {min_val}, Max = {max_val}, Nombre d'observations = {count}")

# Fonction pour charger les données
def load_data(file_path):
    """Charge les données à partir du fichier CSV."""
    return pd.read_csv(file_path)
# Appliquer le GMM et trouver le nombre optimal de clusters (classes)

def apply_gmm(data, column, n_components_range=range(1, 5)):
    """Applique le GMM à une colonne spécifique pour trouver le nombre optimal de classes."""
    X = data[column].values.reshape(-1, 1)  # Redimensionnement pour le GMM
    bics = []
    gmms = []
    
    for n_components in n_components_range:
        gmm = GaussianMixture(n_components=n_components, random_state=42)
        gmm.fit(X)
        gmms.append(gmm)
        bics.append(gmm.bic(X))
    
    optimal_n_components = n_components_range[np.argmin(bics)]
    gmm_optimal = gmms[np.argmin(bics)]
    cluster_labels = gmm_optimal.predict(X)
    
    return optimal_n_components, cluster_labels, n_components_range, bics, gmm_optimal

# Fonction pour créer et afficher les histogrammes
def plot_histograms(data, column, cluster_column):
    """Affiche les histogrammes des classes."""
    plt.figure(figsize=(10, 6))
    sns.histplot(data[cluster_column], bins='auto', kde=False)
    plt.title(f'Distribution des classes pour {column}')
    plt.xlabel('Classe')
    plt.ylabel('Fréquence')
    plt.grid(True)
    plt.show()

# Fonction pour afficher les scores BIC
def plot_bic(n_components_range, bics):
    """Affiche le graphique des scores BIC."""
    plt.figure(figsize=(10, 10))
    plt.plot(n_components_range, bics, marker='o', linestyle='-')
    plt.title('Relation entre le nombre de classes et le BIC')
    plt.xlabel('Nombre de classes')
    plt.ylabel('Score BIC')
    plt.grid(True)
    
    # Annoter chaque point avec son score BIC
    for i, txt in enumerate(bics):
        plt.annotate(f"{txt:.2f}", (n_components_range[i], bics[i]), textcoords="offset points", xytext=(0, 5), ha='center',rotation=45)
    


# Modifier le script principal pour appeler la fonction triée
def main(file_path):
    # Charger les données
    data = load_data(file_path)
    
    # Appliquer le GMM pour la colonne 'radiance'
    optimal_n_components, cluster_labels, n_components_range, bics, gmm_optimal = apply_gmm(data, 'radiance')
    
    # Ajouter les étiquettes de clusters au DataFrame
    data['n_class_repartition'] = cluster_labels
    
    # Afficher le nombre optimal de classes
    print(f"Nombre optimal de classes: {optimal_n_components}")
    
    # Afficher les plages de valeurs des classes triées
    display_class_ranges_sorted(gmm_optimal, data, 'radiance')
    
    # Afficher les scores BIC pour chaque nombre de classes
    plot_bic(n_components_range, bics)
    
    # Afficher les histogrammes des classes
    plot_histograms(data, 'radiance', 'n_class_repartition')
    
    # Afficher le nombre d'observations dans chaque classe
    class_counts = data['n_class_repartition'].value_counts()
    print("\nNombre d'observations par classe:")
    print(class_counts)
    
    return data

In [ ]:
df

In [ ]:
# Lancer l'analyse
file_path = r"D:\df_afrique_pharmacies_schools_combined_sampled_cleaned.csv" # Chemin vers votre fichier CSV 

df_final = main(file_path)

# Afficher les premières lignes du DataFrame final
print(df_final.head())

# Sauvegarder le DataFrame final dans un fichier CSV si nécessaire
df_final.to_csv(r'D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal.csv', index=False)

In [1]:
import pandas as pd

In [21]:
# Je lis la dataframe qui utilise toutes les valeurs incluants les outliers
df=pd.read_csv(r'D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal.csv')

In [ ]:
df.columns

In [ ]:
df['class'].unique()

In [13]:
# Suppression de la colonne 'n_class_repartition'
df = df.drop(columns=['n_class_repartition'])

In [ ]:
df

In [ ]:
import pandas as pd

# Création d'un échantillon de 100000 lignes pour chaque valeur de 'class'
df_class_0 = df[df['class'] == 0].sample(n=100000, random_state=1)
df_class_1 = df[df['class'] == 1].sample(n=100000, random_state=1)
df_class_2 = df[df['class'] == 2].sample(n=100000, random_state=1)
df_class_3 = df[df['class'] == 3].sample(n=100000, random_state=1)

# Concatenation des échantillons pour former le nouveau DataFrame
df_sampled = pd.concat([df_class_0, df_class_1, df_class_2, df_class_3], ignore_index=True)

# Affichage des premières lignes pour vérifier le résultat
print(df_sampled.head(15))
print(df_sampled['class'].value_counts())


In [ ]:
df_sampled

In [ ]:
# Sauvegarde de df_sampled au format CSV
output_path = r"D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_avec_outliers_400k_obs.csv"
df_sampled.to_csv(output_path, index=False)

print(f"Le fichier a été sauvegardé avec succès à l'emplacement : {output_path}")

#### Sans outlier

In [22]:
# Je lis la dataframe qui utilise toutes les valeurs incluants les outliers
df=pd.read_csv(r"D:\df_afrique_pharmacies_schools_combined_sampled_cleaned.csv")

In [ ]:
# Calculer l'IQR pour détecter les outliers
Q1 = df['radiance'].quantile(0.25)
Q3 = df['radiance'].quantile(0.75)
IQR = Q3 - Q1

# Définir les limites pour les outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrer les outliers
outliers = df[(df['radiance'] < lower_bound) | (df['radiance'] > upper_bound)]

# Afficher la latitude, longitude et le pays des outliers
print("Liste des outliers avec leur latitude, longitude, et radiance :")
print(outliers[['latitude', 'longitude','radiance']])


In [ ]:
# Filtrer et conserver uniquement les valeurs non outliers
df = df[(df['radiance'] >= lower_bound) & (df['radiance'] <= upper_bound)]

# Sauvegarde du DataFrame nettoyé
output_path = r"D:\df_afrique_pharmacies_schools_combined_sampled_cleaned_no_outliers.csv"
df.to_csv(output_path, index=False)

print(f"Le DataFrame nettoyé a été sauvegardé sans les outliers à l'emplacement : {output_path}")

In [ ]:
df

Mélange Gaussien

In [36]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns

def display_class_ranges_sorted(gmm, data, column):
    """Affiche les plages de valeurs des classes, triées par valeur minimale."""
    data['class'] = gmm.predict(data[column].values.reshape(-1, 1))
    class_ranges = []

    for i in range(gmm.n_components):
        class_values = data[data['class'] == i][column]
        class_ranges.append((i, class_values.min(), class_values.max(), len(class_values)))

    # Trier les classes par valeur minimale
    class_ranges_sorted = sorted(class_ranges, key=lambda x: x[1])

    # Afficher les informations des classes triées
    for idx, (class_index, min_val, max_val, count) in enumerate(class_ranges_sorted, start=1):
        print(f"Classe {idx}: Min = {min_val}, Max = {max_val}, Nombre d'observations = {count}")

# Fonction pour charger les données
def load_data(file_path):
    """Charge les données à partir du fichier CSV."""
    return pd.read_csv(file_path)
# Appliquer le GMM et trouver le nombre optimal de clusters (classes)

def apply_gmm(data, column, n_components_range=range(1, 5)):
    """Applique le GMM à une colonne spécifique pour trouver le nombre optimal de classes."""
    X = data[column].values.reshape(-1, 1)  # Redimensionnement pour le GMM
    bics = []
    gmms = []
    
    for n_components in n_components_range:
        gmm = GaussianMixture(n_components=n_components, random_state=42)
        gmm.fit(X)
        gmms.append(gmm)
        bics.append(gmm.bic(X))
    
    optimal_n_components = n_components_range[np.argmin(bics)]
    gmm_optimal = gmms[np.argmin(bics)]
    cluster_labels = gmm_optimal.predict(X)
    
    return optimal_n_components, cluster_labels, n_components_range, bics, gmm_optimal

# Fonction pour créer et afficher les histogrammes
def plot_histograms(data, column, cluster_column):
    """Affiche les histogrammes des classes."""
    plt.figure(figsize=(10, 6))
    sns.histplot(data[cluster_column], bins='auto', kde=False)
    plt.title(f'Distribution des classes pour {column}')
    plt.xlabel('Classe')
    plt.ylabel('Fréquence')
    plt.grid(True)
    plt.show()

# Fonction pour afficher les scores BIC
def plot_bic(n_components_range, bics):
    """Affiche le graphique des scores BIC."""
    plt.figure(figsize=(10, 10))
    plt.plot(n_components_range, bics, marker='o', linestyle='-')
    plt.title('Relation entre le nombre de classes et le BIC')
    plt.xlabel('Nombre de classes')
    plt.ylabel('Score BIC')
    plt.grid(True)
    
    # Annoter chaque point avec son score BIC
    for i, txt in enumerate(bics):
        plt.annotate(f"{txt:.2f}", (n_components_range[i], bics[i]), textcoords="offset points", xytext=(0, 5), ha='center',rotation=45)
    


# Modifier le script principal pour appeler la fonction triée
def main(file_path):
    # Charger les données
    data = load_data(file_path)
    
    # Appliquer le GMM pour la colonne 'radiance'
    optimal_n_components, cluster_labels, n_components_range, bics, gmm_optimal = apply_gmm(data, 'radiance')
    
    # Ajouter les étiquettes de clusters au DataFrame
    data['n_class_repartition'] = cluster_labels
    
    # Afficher le nombre optimal de classes
    print(f"Nombre optimal de classes: {optimal_n_components}")
    
    # Afficher les plages de valeurs des classes triées
    display_class_ranges_sorted(gmm_optimal, data, 'radiance')
    
    # Afficher les scores BIC pour chaque nombre de classes
    plot_bic(n_components_range, bics)
    
    # Afficher les histogrammes des classes
    plot_histograms(data, 'radiance', 'n_class_repartition')
    
    # Afficher le nombre d'observations dans chaque classe
    class_counts = data['n_class_repartition'].value_counts()
    print("\nNombre d'observations par classe:")
    print(class_counts)
    
    return data

In [ ]:
# Lancer l'analyse
file_path = r"D:\df_afrique_pharmacies_schools_combined_sampled_cleaned_no_outliers.csv" # Chemin vers le fichier CSV 

df_final = main(file_path)

# Afficher les premières lignes du DataFrame final
print(df_final.head())

# Sauvegarder le DataFrame final dans un fichier CSV si nécessaire
df_final.to_csv(r'D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_no_outliers.csv', index=False)

In [39]:
df=pd.read_csv(r'D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_no_outliers.csv')

In [ ]:
df

In [41]:
# Suppression de la colonne 'n_class_repartition'
df = df.drop(columns=['n_class_repartition'])

In [ ]:
import pandas as pd

# Création d'un échantillon de 80000 lignes pour chaque valeur de 'class'
df_class_0 = df[df['class'] == 0].sample(n=80000, random_state=1)
df_class_1 = df[df['class'] == 1].sample(n=80000, random_state=1)
df_class_2 = df[df['class'] == 2].sample(n=80000, random_state=1)
df_class_3 = df[df['class'] == 3].sample(n=80000, random_state=1)

# Concatenation des échantillons pour former le nouveau DataFrame
df_sampled = pd.concat([df_class_0, df_class_1, df_class_2, df_class_3], ignore_index=True)

# Affichage des premières lignes pour vérifier le résultat
print(df_sampled.head(15))
print(df_sampled['class'].value_counts())


In [ ]:
df_sampled['radiance'].describe()

In [ ]:
# Sauvegarde de df_sampled au format CSV
output_path = r"D:\df_final_afrique_pharmacies_schools_combined_sampled_class_optimal_sans_outliers_320k_obs.csv"
df_sampled.to_csv(output_path, index=False)

print(f"Le fichier a été sauvegardé avec succès à l'emplacement : {output_path}")

In [ ]:
# Filtrage pour obtenir les lignes où la valeur de 'class' est 0 et afficher 10 valeurs
df_class_0_sample = df_sampled[df_sampled['class'] == 0].head(10)

# Affichage des 10 premières valeurs
print("10 valeurs de df_sampled où 'class' est 0 :")
print(df_class_0_sample)


In [ ]:
# Filtrage pour obtenir les lignes où la valeur de 'class' est 1 et afficher 10 valeurs
df_class_1_sample = df_sampled[df_sampled['class'] == 1].head(10)

# Affichage des 10 premières valeurs
print("10 valeurs de df_sampled où 'class' est 1 :")
print(df_class_1_sample)


In [ ]:
# Filtrage pour obtenir les lignes où la valeur de 'class' est 2 et afficher 10 valeurs
df_class_2_sample = df_sampled[df_sampled['class'] == 2].head(10)

# Affichage des 10 premières valeurs
print("10 valeurs de df_sampled où 'class' est 2 :")
print(df_class_2_sample)


In [ ]:
# Filtrage pour obtenir les lignes où la valeur de 'class' est 3 et afficher 10 valeurs
df_class_3_sample = df_sampled[df_sampled['class'] == 3].head(10)

# Affichage des 10 premières valeurs
print("10 valeurs de df_sampled où 'class' est 3 :")
print(df_class_3_sample)
